Step 1 - Import Libraries

In [10]:
import pandas as pd   
import numpy as np     
import os 

Step 2 - Raw Data Structure

In [11]:
df_raw = pd.read_excel('2025.xlsx', sheet_name=0, header=None)

# print the first 5 rows
print(df_raw.head(5))

# print the shape (rows, columns)
print(df_raw.shape)

   0                                           1            2    3    4    5   \
0 NaN  Tourist arrivals from all countries, 2025           NaN  NaN  NaN  NaN   
1 NaN                                         NaN          NaN  NaN  NaN  NaN   
2 NaN                                        Rank      Country  Jan  Feb  Mar   
3 NaN                                           1  Afghanistan    6   15    2   
4 NaN                                           2      Albania   17   33   10   

    6    7    8    9    10   11   12   13   14     15  
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN    NaN  
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN    NaN  
2  Apr  May  Jun  Jul  Aug  Sep  Oct  Nov  Dec  Total  
3   21   10    2    1    3    5    4    3    7     79  
4   14   13    3    8   30   14   10   11   12    175  
(204, 16)


Step 3 - Parse One File

In [12]:
# A list of all 12 month names in order
MONTHS = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

# The first 3 letters of each month in lowercase 
MONTH_SHORT = [m[:3].lower() for m in MONTHS]


def parse_one_file(file_path, year):
    """
    Reads ONE Excel file and returns a clean DataFrame for that year.
    A 'function' is a reusable block of code — instead of writing the same
    logic 7 times (once per file), we write it once and call it 7 times.
    """

    #--Read the file with no header assumption--
    df_raw = pd.read_excel(file_path, sheet_name=0, header=None)

    #--Find the real header row--
    header_row_index = None
    for i in range(len(df_raw)):                        
        row_as_strings = [str(v).strip().lower() for v in df_raw.iloc[i].values]
        if 'country' in row_as_strings:
            header_row_index = i
            break   

    #header row as a list of strings
    header = [str(v).strip() for v in df_raw.iloc[header_row_index].values]

    # --- which column index has "Country" ---
    country_col = None
    for col_idx, value in enumerate(header):
        if value.lower() == 'country':
            country_col = col_idx
            break

    # --- which column index corresponds to each month ---
    month_col_map = {} 
    for col_idx, value in enumerate(header):
        value_lower = value.lower()
        for i, month in enumerate(MONTHS):
            if value_lower.startswith(MONTH_SHORT[i]) and month not in month_col_map:
                month_col_map[month] = col_idx
                break

    # --- Extract the actual data rows ---
    data_rows = []  
    for i in range(header_row_index + 1, len(df_raw)):   
        row_values = df_raw.iloc[i].values                

        country = str(row_values[country_col]).strip()    

        # Skip rows:
        if country in ('nan', '', 'NaN'):   continue   # empty rows
        if len(country) <= 1:               continue   # single-letter separator rows 
        if country.lower() == 'total':      continue   # the grand total row at the bottom

        # Build a dictionary for this row: {'Year': 2019, 'Country': 'India', 'January': 500, ...}
        record = {'Year': year, 'Country': country}

        for month, col_idx in month_col_map.items():
            raw_val = row_values[col_idx]
            try:
                record[month] = int(float(raw_val)) if pd.notna(raw_val) else 0
            except:
                record[month] = 0   

        data_rows.append(record)   

    # Convert the list of dictionaries to a DataFrame
    return pd.DataFrame(data_rows)

Step 4 - Load All 7 Files and Combine

In [13]:
# Dictionary: maps each year to its file name
files = {
    2019: '2019.xlsx',
    2020: '2020.xlsx',
    2021: '2021.xlsx',
    2022: '2022.xlsx',
    2023: '2023.xlsx',
    2024: '2024.xlsx',
    2025: '2025.xlsx',
}

# Loop through each file, parse it, collect results
all_years = []  
for year, filename in files.items():
    df_year = parse_one_file(filename, year)
    print(f"{year}: loaded {len(df_year)} countries")  
    all_years.append(df_year)

# Stack all 7 DataFrames on top of each other into ONE big DataFrame
wide_df = pd.concat(all_years, ignore_index=True)

print(f"\nCombined shape: {wide_df.shape}")
print(wide_df.head(400))

2019: loaded 198 countries
2020: loaded 191 countries
2021: loaded 190 countries
2022: loaded 192 countries
2023: loaded 190 countries
2024: loaded 193 countries
2025: loaded 200 countries

Combined shape: (1354, 14)
     Year      Country  January  February  March  April  May  June  July  \
0    2019  AFGHANISTAN       49        47     59     38    2    12    47   
1    2019      ALBANIA       48        33     54     43   40    31    17   
2    2019      ALGERIA       34        32     36     32    3     8     3   
3    2019      ANDORRA        2         3      5      9    4     0     7   
4    2019       ANGOLA        4         2      0      0    0     0     0   
..    ...          ...      ...       ...    ...    ...  ...   ...   ...   
395  2021    ARGENTINA        0         0      0      5    1     0     1   
396  2021      ARMENIA        2         1      9      5    1     0     0   
397  2021    AUSTRALIA        3        24     29     60   34    13    15   
398  2021      AUSTRIA 

Step 5 - Reshape: Wide → Long Format

In [14]:
long_df = wide_df.melt(
    id_vars=['Year', 'Country'],       
    var_name='Month',                   
    value_name='Number_of_Tourists'     
)

long_df = long_df[long_df['Month'].isin(MONTHS)]

print(f"Long format shape: {long_df.shape}")
print(long_df.head(5))

Long format shape: (16248, 4)
   Year      Country    Month  Number_of_Tourists
0  2019  AFGHANISTAN  January                  49
1  2019      ALBANIA  January                  48
2  2019      ALGERIA  January                  34
3  2019      ANDORRA  January                   2
4  2019       ANGOLA  January                   4


Step 6 - Add the Date Column

In [15]:
# Map month names to numbers: 
month_to_num = {month: i + 1 for i, month in enumerate(MONTHS)}

# Create a Month_Num column 
long_df['Month_Num'] = long_df['Month'].map(month_to_num)

# Build the Date column as "YYYY-MM" string
long_df['Date'] = long_df.apply(
    lambda row: f"{int(row['Year'])}-{int(row['Month_Num']):02d}",
    axis=1
)

# Remove the helper column we no longer need
long_df = long_df.drop(columns=['Month_Num'])

print(long_df[['Year', 'Month', 'Date']].head(5))

   Year    Month     Date
0  2019  January  2019-01
1  2019  January  2019-01
2  2019  January  2019-01
3  2019  January  2019-01
4  2019  January  2019-01


Step 7 — Clean the Data

In [16]:
print("=== BEFORE CLEANING ===")
print(f"Rows: {len(long_df)}")
print(f"Missing values:\n{long_df.isnull().sum()}")
print(f"Duplicates: {long_df.duplicated(subset=['Year', 'Month', 'Country']).sum()}")

# 1. Fill missing tourist numbers with 0
long_df['Number_of_Tourists'] = long_df['Number_of_Tourists'].fillna(0).astype(int)

# 2. Remove duplicates
long_df = long_df.drop_duplicates(subset=['Year', 'Month', 'Country'])

# 3. Standardize country names to Title Case
long_df['Country'] = long_df['Country'].str.title()

# 4. Sort by Date then Country 
long_df = long_df.sort_values(['Date', 'Country']).reset_index(drop=True)

# 5. Final column order
long_df = long_df[['Year', 'Month', 'Country', 'Number_of_Tourists', 'Date']]

print("\n=== AFTER CLEANING ===")
print(f"Rows: {len(long_df)}")
print(f"Missing values:\n{long_df.isnull().sum()}")
print(f"Duplicates: {long_df.duplicated(subset=['Year', 'Month', 'Country']).sum()}")
print(f"Date range: {long_df['Date'].min()} → {long_df['Date'].max()}")
print(f"Unique countries: {long_df['Country'].nunique()}")
print(f"\nFinal sample:\n{long_df.head(10)}")

=== BEFORE CLEANING ===
Rows: 16248
Missing values:
Year                  0
Country               0
Month                 0
Number_of_Tourists    0
Date                  0
dtype: int64
Duplicates: 180

=== AFTER CLEANING ===
Rows: 16068
Missing values:
Year                  0
Month                 0
Country               0
Number_of_Tourists    0
Date                  0
dtype: int64
Duplicates: 0
Date range: 2019-01 → 2025-12
Unique countries: 243

Final sample:
   Year    Month              Country  Number_of_Tourists     Date
0  2019  January          Afghanistan                  49  2019-01
1  2019  January              Albania                  48  2019-01
2  2019  January              Algeria                  34  2019-01
3  2019  January              Andorra                   2  2019-01
4  2019  January               Angola                   4  2019-01
5  2019  January  Antigua And Barbuda                   1  2019-01
6  2019  January            Argentina                 192  2019-

Step 8 — Save the Clean File

In [17]:
long_df.to_csv('sl_tourism_clean.csv', index=False)

print("✅ Saved: sl_tourism_clean.csv")
print(f"Total rows: {len(long_df)}")
print("This file is ready for Google Colab, Power BI, or SQL.")

✅ Saved: sl_tourism_clean.csv
Total rows: 16068
This file is ready for Google Colab, Power BI, or SQL.
